In [ ]:
pip install -q docling

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 537.8/537.8 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.0/297.0 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.0/94.0 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 98.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 72.4 MB/s eta 0:00:00
   ━━

In [ ]:
from docling.document_converter import DocumentConverter
source = "/content/mrag-research-paper"
converter = DocumentConverter()
doc = converter.convert(source).document

[INFO] 2026-06-11 17:35:42,974 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-06-11 17:35:42,984 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-06-11 17:35:42,991 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.8.0/torch/PP-OCRv4/det/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-06-11 17:35:43,871 [RapidOCR] download_file.py:82: Download size: 13.83MB
[INFO] 2026-06-11 17:35:44,257 [RapidOCR] download_file.py:95: Successfully saved to: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-06-11 17:35:44,259 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-06-11 17:35:44,600 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-06-11 17:35:44,601 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-06-11 17:35:44,603 [RapidOCR] download_file.py:68: Initiating download: https

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

In [ ]:
import json
import hashlib
from pathlib import Path

from docling_core.transforms.chunker import HierarchicalChunker

# Create output directory
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

# Chunk document
chunker = HierarchicalChunker()
chunks = [chunk.text for chunk in chunker.chunk(doc)]

# Create JSON structure
chunk_json = {}

for chunk in chunks:
    chunk_hash = hashlib.sha256(
        chunk.encode("utf-8")
    ).hexdigest()

    chunk_json[chunk_hash] = chunk

# Save JSON
json_path = output_dir / "chunks.json"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(
        chunk_json,
        f,
        ensure_ascii=False,
        indent=2
    )

print(f"Saved {len(chunks)} chunks to {json_path}")

Saved 101 chunks to output/chunks.json


In [ ]:
pip install -q PyMuPDF ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.6 MB/s eta 0:00:00


In [ ]:
import os
import json
import fitz
from PIL import Image
from ultralytics import YOLO

# ─── CONFIG ───────────────────────────────────────────────
INPUT_FILE    = "/content/mrag-research-paper"
OUTPUT_DIR    = "output/images"
JSON_FILE     = "output/image_text.json"
MODEL_PATH    = "/content/yolov8s-doclaynet.pt"
DPI           = 200
CONFIDENCE    = 0.5
VICINITY      = 50    # PDF units — how far "nearby" means
# ──────────────────────────────────────────────────────────

FIGURE_LABELS = {"picture", "figure"}

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs("output", exist_ok=True)

model = YOLO(MODEL_PATH)

doc   = fitz.open(INPUT_FILE)
zoom  = DPI / 72
mat   = fitz.Matrix(zoom, zoom)
total = 0

result_json = {}  # { image_path: nearby_text }

for page_num in range(len(doc)):
    page = doc[page_num]
    pix  = page.get_pixmap(matrix=mat)
    page_image = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

    results = model(page_image, conf=CONFIDENCE)[0]

    fig_count = 0
    for box, cls, conf in zip(results.boxes.xyxy, results.boxes.cls, results.boxes.conf):
        label = model.names[int(cls)].lower()
        if label not in FIGURE_LABELS:
            continue

        # ── Save image ──────────────────────────────────────
        x1, y1, x2, y2 = [int(c) for c in box]
        cropped  = page_image.crop((x1, y1, x2, y2))
        filename = f"page{page_num + 1}_figure{fig_count + 1}.png"
        img_path = os.path.join(OUTPUT_DIR, filename)
        cropped.save(img_path)

        # ── Get nearby text ──────────────────────────────────
        # Convert pixel bbox back to PDF units for text search
        pdf_x1 = x1 / zoom
        pdf_y1 = y1 / zoom
        pdf_x2 = x2 / zoom
        pdf_y2 = y2 / zoom

        # Expand the box by VICINITY to capture surrounding text
        search_rect = fitz.Rect(
            pdf_x1 - VICINITY,
            pdf_y1 - VICINITY,
            pdf_x2 + VICINITY,
            pdf_y2 + VICINITY
        )

        # Extract text only within that expanded region
        nearby_text = page.get_text("text", clip=search_rect).strip()

        # ── Store in JSON ────────────────────────────────────
        result_json[img_path] = nearby_text

        print(f"  Saved : {filename}  (conf: {conf:.2f})")
        print(f"  Text  : {nearby_text[:80]}{'...' if len(nearby_text) > 80 else ''}")

        fig_count += 1
        total += 1

    print(f"Page {page_num + 1}: {fig_count} figure(s) found\n")

doc.close()

# ── Write JSON ───────────────────────────────────────────
with open(JSON_FILE, "w", encoding="utf-8") as f:
    json.dump(result_json, f, indent=4, ensure_ascii=False)

print(f"[✓] Done! {total} images saved to : {OUTPUT_DIR}/")
print(f"[✓] JSON saved to                 : {JSON_FILE}")


0: 1024x800 1 Page-footer, 4 Section-headers, 17 Texts, 1 Title, 1082.5ms
Speed: 10.8ms preprocess, 1082.5ms inference, 16.4ms postprocess per image at shape (1, 3, 1024, 800)
Page 1: 0 figure(s) found


0: 1024x800 4 List-items, 1 Page-footer, 1 Page-header, 1 Picture, 7 Texts, 1042.6ms
Speed: 7.8ms preprocess, 1042.6ms inference, 1.1ms postprocess per image at shape (1, 3, 1024, 800)
  Saved : page2_figure1.png  (conf: 0.87)
  Text  : SIGIR ’25, July 13–18, 2025, Padua, Italy
Qinhan Yu et al.
The left is a Bengal ...
Page 2: 1 figure(s) found


0: 1024x800 3 List-items, 1 Page-footer, 6 Section-headers, 9 Texts, 1055.2ms
Speed: 8.0ms preprocess, 1055.2ms inference, 1.1ms postprocess per image at shape (1, 3, 1024, 800)
Page 3: 0 figure(s) found


0: 1024x800 1 Page-footer, 1 Picture, 2 Section-headers, 1 Table, 6 Texts, 1031.2ms
Speed: 8.0ms preprocess, 1031.2ms inference, 1.1ms postprocess per image at shape (1, 3, 1024, 800)
  Saved : page4_figure1.png  (conf: 0.82)
  Text  : WebQ

In [ ]:
pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.6 MB/s eta 0:00:00


In [ ]:
import os
import json
import hashlib
from groq import Groq

# ─── CONFIG ───────────────────────────────────────────────
INPUT_JSON  = "output/image_text.json"
OUTPUT_JSON = "output/image_descriptions.json"
client = Groq(api_key="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")
# ──────────────────────────────────────────────────────────

def image_to_base64_for_groq(image_path):
    import base64
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def sha256_hash(image_path):
    with open(image_path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()

def generate_description(image_base64, nearby_text):
    prompt = f"""
You are extracting information from a PDF.
Nearby text:
{nearby_text}
Task:
Generate a concise, information-dense description of the image.
Use nearby text only when relevant.
Return only the description.
"""
    completion = client.chat.completions.create(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"},
                    },
                ],
            }
        ],
        temperature=0.2,
        max_completion_tokens=300,
    )
    return completion.choices[0].message.content

# ─── MAIN ─────────────────────────────────────────────────
with open(INPUT_JSON, "r") as f:
    image_text = json.load(f)

output = {}

for img_path, nearby_text in image_text.items():
    if not os.path.exists(img_path):
        print(f"  Skipping (not found): {img_path}")
        continue

    print(f"Processing: {img_path}")

    b64         = image_to_base64_for_groq(img_path)   # only for API call
    sha         = sha256_hash(img_path)
    description = generate_description(b64, nearby_text)

    output[sha] = [img_path, description]              # path instead of base64
    print(f"  Hash : {sha[:20]}...")
    print(f"  Desc : {description[:80]}...")

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=4, ensure_ascii=False)

print(f"\n[✓] Done! Saved to: {OUTPUT_JSON}")

Processing: output/images/page2_figure1.png
  Hash : 7f605d9a361c910613ae...
  Desc : The image presents a comprehensive overview of a multimodal database system, sho...
Processing: output/images/page4_figure1.png
  Hash : afead67c98e98342c36c...
  Desc : The MRAMG-Bench construction pipeline consists of three stages: Data Selection a...
Processing: output/images/page7_figure1.png
  Hash : a4f77d451c27966e6e00...
  Desc : The image presents a comparison of two algorithms, Graph Matching and Max Simila...

[✓] Done! Saved to: output/image_descriptions.json


In [ ]:
import json
from pathlib import Path
from sentence_transformers import SentenceTransformer, util

# -------- Paths --------
output_dir = Path("output")
chunks_path = output_dir / "chunks.json"
images_path = output_dir / "image_descriptions.json"
merged_path = output_dir / "merged_chunks_images.json"

# -------- Load JSON files --------
with open(chunks_path, "r", encoding="utf-8") as f:
    chunks_json = json.load(f)

with open(images_path, "r", encoding="utf-8") as f:
    images_json = json.load(f)

# -------- Load CLIP text embedding model --------
model = SentenceTransformer("clip-ViT-B-32")

# -------- Prepare data --------
chunk_ids   = list(chunks_json.keys())
chunk_texts = list(chunks_json.values())

image_ids         = list(images_json.keys())
image_descriptions = [images_json[img_id][1] for img_id in image_ids]

# -------- Create embeddings --------
chunk_embeddings = model.encode(chunk_texts, convert_to_tensor=True)
image_embeddings = model.encode(image_descriptions, convert_to_tensor=True)

# -------- Similarity threshold --------
SIMILARITY_THRESHOLD = 0.8   # CLIP scores are lower than MiniLM — adjust accordingly

merged_json = {}

# -------- Compare every chunk with every image description --------
for i, chunk_id in enumerate(chunk_ids):
    for j, image_id in enumerate(image_ids):

        similarity = util.cos_sim(
            chunk_embeddings[i],
            image_embeddings[j]
        ).item()

        if similarity >= SIMILARITY_THRESHOLD:
            key = f"({chunk_id}, {image_id})"
            merged_json[key] = {
                "chunk_id"  : chunk_id,
                "image_id"  : image_id,
                "similarity": round(similarity, 4),
                "content"   : chunks_json[chunk_id] + "\n\n" + image_descriptions[j]
            }

# -------- Save new JSON --------
with open(merged_path, "w", encoding="utf-8") as f:
    json.dump(merged_json, f, ensure_ascii=False, indent=2)

print(f"Saved {len(merged_json)} merged records to {merged_path}")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Saved 42 merged records to output/merged_chunks_images.json


In [ ]:
pip install -q chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the s

In [ ]:
import json
from pathlib import Path

import chromadb
from sentence_transformers import SentenceTransformer


# -------- Paths --------
output_dir = Path("output")

chunks_path = output_dir / "chunks.json"
merged_path = output_dir / "merged_chunks_images.json"

chroma_path = "chroma_db"


# -------- Load JSON files --------
with open(chunks_path, "r", encoding="utf-8") as f:
    chunks_json = json.load(f)

with open(merged_path, "r", encoding="utf-8") as f:
    merged_json = json.load(f)


# -------- Embedding model --------
model = SentenceTransformer("clip-ViT-B-32")


# -------- ChromaDB setup --------
client = chromadb.PersistentClient(path=chroma_path)

collection = client.get_or_create_collection(
    name="pdf_chunks"
)


# -------- Prepare records --------
ids = []
documents = []
metadatas = []

# 1. Add normal chunks
for chunk_id, chunk_text in chunks_json.items():
    ids.append(chunk_id)
    documents.append(chunk_text)
    metadatas.append({
        "type": "text_chunk",
        "chunk_id": chunk_id
    })

# 2. Add merged chunk + image descriptions
for merged_id, merged_data in merged_json.items():

    # If your merged JSON value is a dict
    if isinstance(merged_data, dict):
        text = merged_data["content"]
        chunk_id = merged_data.get("chunk_id")
        image_id = merged_data.get("image_id")
        similarity = merged_data.get("similarity")

    # If your merged JSON value is only plain text
    else:
        text = merged_data
        chunk_id = None
        image_id = None
        similarity = None

    ids.append(merged_id)
    documents.append(text)
    metadatas.append({
        "type": "merged_chunk_image",
        "chunk_id": chunk_id or "",
        "image_id": image_id or "",
        "similarity": similarity or 0
    })


# -------- Create embeddings --------
embeddings = model.encode(
    documents,
    convert_to_numpy=True
).tolist()


# -------- Save to ChromaDB --------
collection.add(
    ids=ids,
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas
)

print(f"Saved {len(ids)} records to ChromaDB")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Saved 142 records to ChromaDB


In [ ]:
data = collection.get()

import pandas as pd

df = pd.DataFrame({
    "id": data["ids"],
    "document": data["documents"],
    "metadata": data["metadatas"]
})

print(df.head())

                                                  id  \
0  f315cd4386a928d2fd925ea148f03524c8bbd6fb6fd712...   
1  2e9f0a0474d0a41cdcefe21092139c2c381b36dc41d2a6...   
2  311f2b12e69b25ce67f2f4c37f33b44b2dfe0bd07a08ab...   
3  34503369fd223998fe7772e5cd1ef17c7e1ec18db1ad01...   
4  b6491437994cd2545463e30e7d261fb4be5026eba0ca9a...   

                                            document  \
0  Qinhan Yu ∗†, 1 = Zhiyou Xiao ∗. Qinhan Yu ∗†,...   
1  Recent advances in Retrieval-Augmented Generat...   
2  ∗ These authors contributed equally to this work.   
3        † Work done during an internship at Huawei.   
4                           ‡ Corresponding Authors.   

                                            metadata  
0  {'chunk_id': 'f315cd4386a928d2fd925ea148f03524...  
1  {'chunk_id': '2e9f0a0474d0a41cdcefe21092139c2c...  
2  {'type': 'text_chunk', 'chunk_id': '311f2b12e6...  
3  {'type': 'text_chunk', 'chunk_id': '34503369fd...  
4  {'chunk_id': 'b6491437994cd2545463e30e7d261fb4..

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb

# Load embedding model
model = SentenceTransformer("clip-ViT-B-32")

# Load ChromaDB
client = chromadb.PersistentClient(path="chroma_db")

collection = client.get_collection("pdf_chunks")

# Query
query = "Explain me all three stage in the RAG pipeline in the given image"

# Encode query
query_embedding = model.encode(
    query,
    convert_to_numpy=True
).tolist()

# Retrieve top 10 most similar documents
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=10
)

# Print results
for rank, (doc_id, doc, distance) in enumerate(
    zip(
        results["ids"][0],
        results["documents"][0],
        results["distances"][0]
    ),
    start=1
):
    print(f"\nRank: {rank}")
    print(f"ID: {doc_id}")
    print(f"Distance: {distance}")
    print(f"Document: {doc[:500]}...")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]


Rank: 1
ID: 189f30aa2b6e7ed49f454ece90d8060a059e9e2527ec6d67e47fa774f18c750b
Distance: 35.71117401123047
Document: This stage can be further divided into three steps: (1) Question Generation, (2) Answer Generation, (3) QA Pair Refinement....

Rank: 2
ID: 25a2254d24f13bbd1396f04bff8994b466b7719e8b017de641e621b5a2c9da59
Distance: 42.070396423339844
Document: Why not present the images directly instead of describing their content?...

Rank: 3
ID: b66f93a34b3fa22efc6015858543faef887772d84990522d99c7911fa8dccf85
Distance: 42.391212463378906
Document: The effective evaluation of Retrieval-Augmented Generation (RAG) models is crucial for advancing their development and optimization. Established benchmarks [17, 19, 27, 33, 46] are commonly used to evaluate RAG models [31, 42]. MS-MARCO [27] is a large-scale question answering (QA) dataset based on real user queries. TriviaQA [17] consists of trivia questions that require evidence-based answers. HotpotQA [46] focuses on multi-hop reasoning. Na

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "BAAI/bge-reranker-base"
)

pairs = [
    [query, doc]
    for doc in results["documents"][0]
]

scores = reranker.predict(pairs)

reranked_results = sorted(
    zip(
        results["ids"][0],
        results["documents"][0],
        scores
    ),
    key=lambda x: x[2],
    reverse=True
)

top_3 = reranked_results[:5]

for rank, (doc_id, doc, score) in enumerate(top_3, start=1):
    print(f"\nRank: {rank}")
    print(f"ID: {doc_id}")
    print(f"Score: {score:.4f}")
    print(f"Document: {doc[:500]}...")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Rank: 1
ID: 189f30aa2b6e7ed49f454ece90d8060a059e9e2527ec6d67e47fa774f18c750b
Score: 0.0979
Document: This stage can be further divided into three steps: (1) Question Generation, (2) Answer Generation, (3) QA Pair Refinement....

Rank: 2
ID: b66f93a34b3fa22efc6015858543faef887772d84990522d99c7911fa8dccf85
Score: 0.0217
Document: The effective evaluation of Retrieval-Augmented Generation (RAG) models is crucial for advancing their development and optimization. Established benchmarks [17, 19, 27, 33, 46] are commonly used to evaluate RAG models [31, 42]. MS-MARCO [27] is a large-scale question answering (QA) dataset based on real user queries. TriviaQA [17] consists of trivia questions that require evidence-based answers. HotpotQA [46] focuses on multi-hop reasoning. Natural Questions (NQ) [19] is derived from real Google...

Rank: 3
ID: 82e0bb97830f214d29a232311060191735a20919e18e2ba42343dd3fcf3afa9d
Score: 0.0129
Document: Experiment Details. In the multimodal documents, images are rep

In [ ]:
import json
from pathlib import Path

# --------------------------
# Load JSON files
# --------------------------
output_dir = Path("output")

with open(output_dir / "chunks.json", "r", encoding="utf-8") as f:
    chunks_json = json.load(f)

with open(output_dir / "image_descriptions.json", "r", encoding="utf-8") as f:
    images_json = json.load(f)

# --------------------------
# Collect all unique IDs
# --------------------------
all_ids = set()

for doc_id, _, _ in top_3:

    # Merged ID format:
    # (chunk_hash, image_hash)
    if doc_id.startswith("(") and doc_id.endswith(")"):

        ids_str = doc_id[1:-1]  # remove '(' and ')'

        try:
            chunk_id, image_id = [
                x.strip()
                for x in ids_str.split(",", maxsplit=1)
            ]

            all_ids.add(chunk_id)
            all_ids.add(image_id)

        except ValueError:
            print(f"Could not parse merged ID: {doc_id}")

    else:
        all_ids.add(doc_id)

# --------------------------
# Retrieve matching chunks/images
# --------------------------
retrieved_chunks = {}
retrieved_images = {}

for item_id in all_ids:

    if item_id in chunks_json:
        retrieved_chunks[item_id] = chunks_json[item_id]

    if item_id in images_json:
        # [base64, description]
        retrieved_images[item_id] = {
            "base64": images_json[item_id][0],
            "description": images_json[item_id][1]
        }

# --------------------------
# Print text chunks
# --------------------------
print("\nTEXT CHUNKS")
print("=" * 100)

for chunk_id, chunk_text in retrieved_chunks.items():

    print(f"\nChunk ID:\n{chunk_id}")
    print("-" * 100)
    print(chunk_text[:1000])

# --------------------------
# Print image information
# --------------------------
print("\nIMAGES")
print("=" * 100)

for image_id, image_data in retrieved_images.items():

    print(f"\nImage ID:\n{image_id}")
    print("-" * 100)

    print("\nDescription:")
    print(image_data["description"])

    print("\nBase64 (first 200 chars):")
    print(image_data["base64"][:200] + "...")

    print("\n" + "=" * 100)


TEXT CHUNKS

Chunk ID:
b36f9025608d998ceae1ed299c4aa856dc58e46853a41acfcd4f8c031aeb8743
----------------------------------------------------------------------------------------------------
- Images as answers : ' Show, don't tell . ' In certain cases, an image provides the most effective response. For instance, when asked about the appearance of a cat, a photograph is the most direct and convincing answer-seeing truly is believing.
- Text-Image integration for answers : ' Apicture and words are two puzzle pieces-only together do they reveal the full story. ' Integrating pictures and words, such as in step-by-step recipe instructions, significantly enhances comprehension which offers clearer insights and fosters a more intuitive understanding of the content.
- Images as a catalyst for richer answer : ' A picture is the icing on the cake. ' In applications such as tourist spot descriptions, integrating textual and visual content can significantly enhance the quality of the generated res

In [ ]:
pip install -q langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.2 MB/s eta 0:00:00


In [ ]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
import os
import base64

llm = ChatGroq(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    api_key="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",
    temperature=0.2,
    max_tokens=700
)

# --------------------------
# Prepare text context
# --------------------------
text_context = ""

for chunk_id, chunk_text in retrieved_chunks.items():
    text_context += f"\n\nChunk ID: {chunk_id}\n{chunk_text}"

# --------------------------
# Prepare multimodal message
# --------------------------
content = [
    {
        "type": "text",
        "text": f"""
You are answering a user query using retrieved PDF text chunks and related images.

User query:
{query}

Retrieved text chunks:
{text_context}

Instructions:
- Answer only from the provided text and images.
- Use images only when relevant.
- If the answer is not present, say that the provided context is insufficient.
- Be clear and concise.
"""
    }
]

def path_to_base64(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

# Add images
for image_id, image_data in list(retrieved_images.items())[:5]:
    image_base64 = path_to_base64(image_data["base64"])  # "base64" key is actually the path

    content.append(
        {
            "type": "image_url",
            "image_url": {
                "url": f"data:image/jpeg;base64,{image_base64}"
            }
        }
    )

# --------------------------
# Invoke Vision LLM
# --------------------------
response = llm.invoke([
    HumanMessage(content=content)
])
answer = response.content
print(response.content)

The three stages in the RAG pipeline depicted in the image are:

## Stage 1: Retrieval 
The retrieval stage involves searching a multimodal database using a user's query. This stage retrieves relevant text and images.

## Stage 2: Generation 
In the generation stage, a generative model uses the retrieved text and images to create a response.

## Stage 3: Response 
The final stage presents the generated multimodal answer to the user. 

However, another text chunk (ID: 189f30aa2b6e7ed49f454ece90d8060a059e9e2527ec6d67e47fa774f18c750b) divides the RAG pipeline into three steps: 
1. Question Generation
2. Answer Generation
3. QA Pair Refinement. 

These steps seem to be related to the internal process of the pipeline rather than the overall stages. 

Based on the provided image and text chunks, the three stages in the RAG pipeline appear to be retrieval, generation, and response.


In [ ]:
pip install -q networkx sentence-transformers

In [ ]:
import networkx as nx
from sentence_transformers import SentenceTransformer, util
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

model = SentenceTransformer("clip-ViT-B-32")

# sentences = answer.split(". ")
sentences = sent_tokenize(answer)
image_ids = list(retrieved_images.keys())
image_descriptions = [
    retrieved_images[img_id]["description"]
    for img_id in image_ids
]

sentence_embeddings = model.encode(sentences, convert_to_tensor=True)
image_embeddings = model.encode(image_descriptions, convert_to_tensor=True)

G = nx.Graph()

for i, sentence in enumerate(sentences):
    for j, image_id in enumerate(image_ids):
        similarity = util.cos_sim(
            sentence_embeddings[i],
            image_embeddings[j]
        ).item()

        if similarity > 0.2:
            G.add_edge(
                f"s_{i}",
                f"img_{j}",
                weight=similarity
            )

matching = nx.algorithms.matching.max_weight_matching(
    G,
    maxcardinality=False
)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [ ]:
sentence_to_image = {}

for a, b in matching:
    if a.startswith("s_"):
        sentence_id = int(a.split("_")[1])
        image_id = image_ids[int(b.split("_")[1])]
    else:
        sentence_id = int(b.split("_")[1])
        image_id = image_ids[int(a.split("_")[1])]

    sentence_to_image[sentence_id] = image_id

In [ ]:
for i, sentence in enumerate(sentences):
    print(sentence)

    if i in sentence_to_image:
        img_id = sentence_to_image[i]
        print(f"\n[Insert image: {img_id}]\n")

The three stages in the RAG pipeline depicted in the image are:

## Stage 1: Retrieval 
The retrieval stage involves searching a multimodal database using a user's query.

[Insert image: 7f605d9a361c910613ae20c670786052d88f856bbe6c3e129f252e13c436fac7]

This stage retrieves relevant text and images.
## Stage 2: Generation 
In the generation stage, a generative model uses the retrieved text and images to create a response.
## Stage 3: Response 
The final stage presents the generated multimodal answer to the user.
However, another text chunk (ID: 189f30aa2b6e7ed49f454ece90d8060a059e9e2527ec6d67e47fa774f18c750b) divides the RAG pipeline into three steps: 
1.
Question Generation
2.
Answer Generation
3.
QA Pair Refinement.
These steps seem to be related to the internal process of the pipeline rather than the overall stages.
Based on the provided image and text chunks, the three stages in the RAG pipeline appear to be retrieval, generation, and response.
